# Interactive Single-Source Matching Notebook
Use this notebook to pick one source account, choose a date range, run deterministic matching against YNAB, and inspect matched/unmatched records.

Workflow:
1. Run setup/imports.
2. Load data from pipeline or CSV.
3. Clean/parse columns.
4. Use controls and click **Run Matching**.
5. Review table + optional map and export CSV outputs.

In [12]:
import pathlib
from datetime import datetime
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

from etl_pipeline.consolidate import build_master_df
from etl_pipeline.matching import run_cascade_matching, MatchLevel, compute_match_score

try:
    import folium
    from folium.plugins import MarkerCluster
    HAS_FOLIUM = True
except Exception:
    HAS_FOLIUM = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

RAW_DF = None
CLEAN_DF = None
LAST_FILTERED = None
LAST_MATCHED = None
LAT_COL = None
LON_COL = None

## 1) Set Up Environment and Imports
Core libraries are loaded, plus your deterministic ETL matching functions. A map library is optional (Folium).

## 2) Load the Single Source Dataset
Load from either:
- Pipeline output (recommended): builds consolidated data from configured folders.
- Existing CSV file path.

This step also validates required matching columns.

In [13]:
REQUIRED_COLS = ["Date", "Account", "Inflow", "Outflow", "source_type", "Payee", "Memo"]

def load_dataset(mode="pipeline", csv_path=None, dates_range=None):
    global RAW_DF
    if mode == "pipeline":
        RAW_DF = build_master_df(dates_range=dates_range, fail_on_error=True).copy()
    elif mode == "csv":
        if not csv_path:
            raise ValueError("csv_path is required when mode='csv'")
        RAW_DF = pd.read_csv(csv_path).copy()
    else:
        raise ValueError("mode must be 'pipeline' or 'csv'")

    missing = [c for c in REQUIRED_COLS if c not in RAW_DF.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    print(f"Loaded rows: {len(RAW_DF)}")
    print(f"Columns: {list(RAW_DF.columns)}")
    display(RAW_DF.head(10))
    return RAW_DF

# Example 1: load from pipeline across all dates
_ = load_dataset(mode="pipeline", dates_range=None)

# Example 2: uncomment to load from CSV
# _ = load_dataset(mode="csv", csv_path="private_data/outgoing/master_20260509_225723.csv")

[Mizrachi] loading from /Users/shai/Documents/Projects/Ynab personal consolidator israeli accounts/private_data/incoming/mizrachi_joint ...
[Mizrachi] AccountActivity.xls: 44 rows
[Bank Leumi] loading from /Users/shai/Documents/Projects/Ynab personal consolidator israeli accounts/private_data/incoming/bank_leumi_private_shai ...
[Bank Leumi] תנועות בחשבון 3_5_2026.xls: 4 rows
[Bank Hapoalim] loading from /Users/shai/Documents/Projects/Ynab personal consolidator israeli accounts/private_data/incoming/bank_hapoalim_private_shai ...
[Bank Hapoalim] excelNewTransactions (1).xlsx: 37 rows
[Max Uniq] loading from /Users/shai/Documents/Projects/Ynab personal consolidator israeli accounts/private_data/incoming/max_uniq_joint ...
[Max Uniq] transaction-details_export_1777835474454.xlsx: 26 rows
[Max Uniq] transaction-details_export_1777835474454.xlsx: 2 rows
[Max Uniq] transaction-details_export_1777835484434.xlsx: 31 rows
[Max Uniq] transaction-details_export_1777835484434.xlsx: 1 rows
[Visa C

/Users/shai/anaconda3/envs/ynab-consolidator/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/Users/shai/anaconda3/envs/ynab-consolidator/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Date,Payee,Memo,Inflow,Outflow,Account,Ownership,source_type
0,04/02/2026,זיכוי - בנק הפועלים(י),,5000.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card
1,06/02/2026,הרשאה מקס איט פיננס(י),,0.0,62.78,Mizrachi,Shai & Nirit (Joint),bank_card
2,08/02/2026,זיכוי - בנק הפועלים(י),,17000.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card
3,10/02/2026,ביטוח לאומי חד פעמי(י),,20315.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card
4,10/02/2026,ביצוע הוראת קבע (י),,0.0,600.00,Mizrachi,Shai & Nirit (Joint),bank_card
5,10/02/2026,הרשאה ישראכרט (י),,0.0,3427.00,Mizrachi,Shai & Nirit (Joint),bank_card
6,10/02/2026,הרשאה מקס איט פיננס(י),,0.0,500.00,Mizrachi,Shai & Nirit (Joint),bank_card
7,10/02/2026,"מור גמל ופנסיה בע""מ(י)",,0.0,800.00,Mizrachi,Shai & Nirit (Joint),bank_card
8,13/02/2026,ביטוח לאומי חד פעמי(י),,946.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card
9,13/02/2026,העברה באינטרנט (י),,0.0,2000.00,Mizrachi,Shai & Nirit (Joint),bank_card


## 3) Clean Fields and Parse Date/Location Columns
Standardize types, parse dates, and discover optional latitude/longitude columns for mapping.

In [14]:
def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    global LAT_COL, LON_COL
    out = df.copy()

    out.columns = [c.strip() for c in out.columns]
    out["Memo"] = out["Memo"].fillna("").astype(str)
    out["Payee"] = out["Payee"].fillna("").astype(str)
    out["Account"] = out["Account"].fillna("").astype(str)
    out["source_type"] = out["source_type"].fillna("").astype(str)

    out["Date_dt"] = pd.to_datetime(out["Date"], format="%d/%m/%Y", errors="coerce")
    out["Inflow"] = pd.to_numeric(out["Inflow"], errors="coerce").fillna(0.0).round(2)
    out["Outflow"] = pd.to_numeric(out["Outflow"], errors="coerce").fillna(0.0).round(2)
    out["amount_abs"] = (out["Inflow"] + out["Outflow"]).abs()
    out["direction"] = np.where(out["Outflow"] > 0, "outflow", np.where(out["Inflow"] > 0, "inflow", "zero"))

    lat_candidates = ["lat", "latitude", "Latitude", "LAT"]
    lon_candidates = ["lon", "lng", "longitude", "Longitude", "LON", "LNG"]
    LAT_COL = next((c for c in lat_candidates if c in out.columns), None)
    LON_COL = next((c for c in lon_candidates if c in out.columns), None)

    if LAT_COL and LON_COL:
        out[LAT_COL] = pd.to_numeric(out[LAT_COL], errors="coerce")
        out[LON_COL] = pd.to_numeric(out[LON_COL], errors="coerce")
        coord_mask = out[LAT_COL].between(-90, 90) & out[LON_COL].between(-180, 180)
        out.loc[~coord_mask, [LAT_COL, LON_COL]] = np.nan

    return out

if RAW_DF is None:
    raise RuntimeError("Run the data loading cell first.")

CLEAN_DF = prepare_dataframe(RAW_DF)
print(f"Prepared rows: {len(CLEAN_DF)}")
print(f"Parsed dates (non-null): {CLEAN_DF['Date_dt'].notna().sum()}")
print(f"Lat/Lon columns found: {LAT_COL}, {LON_COL}")
display(CLEAN_DF.head(10))

Prepared rows: 6069
Parsed dates (non-null): 6069
Lat/Lon columns found: None, None


,Date,Payee,Memo,Inflow,Outflow,Account,Ownership,source_type,Date_dt,amount_abs,direction
0,04/02/2026,זיכוי - בנק הפועלים(י),,5000.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-04,5000.00,inflow
1,06/02/2026,הרשאה מקס איט פיננס(י),,0.0,62.78,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-06,62.78,outflow
2,08/02/2026,זיכוי - בנק הפועלים(י),,17000.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-08,17000.00,inflow
3,10/02/2026,ביטוח לאומי חד פעמי(י),,20315.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-10,20315.00,inflow
4,10/02/2026,ביצוע הוראת קבע (י),,0.0,600.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-10,600.00,outflow
5,10/02/2026,הרשאה ישראכרט (י),,0.0,3427.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-10,3427.00,outflow
6,10/02/2026,הרשאה מקס איט פיננס(י),,0.0,500.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-10,500.00,outflow
7,10/02/2026,"מור גמל ופנסיה בע""מ(י)",,0.0,800.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-10,800.00,outflow
8,13/02/2026,ביטוח לאומי חד פעמי(י),,946.0,0.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-13,946.00,inflow
9,13/02/2026,העברה באינטרנט (י),,0.0,2000.00,Mizrachi,Shai & Nirit (Joint),bank_card,2026-02-13,2000.00,outflow


## 4) Create Interactive Date Range Controls
Pick one source account, date range, and matching tolerances.

In [ ]:
if CLEAN_DF is None:
    raise RuntimeError("Run the cleaning cell first.")

date_min = CLEAN_DF["Date_dt"].min()
date_max = CLEAN_DF["Date_dt"].max()
account_options = sorted([a for a in CLEAN_DF["Account"].dropna().unique().tolist() if str(a).strip()])

source_dropdown = widgets.Dropdown(
    options=account_options,
    description="Source:",
    layout=widgets.Layout(width="420px")
)
start_picker = widgets.DatePicker(
    description="From:",
    value=None if pd.isna(date_min) else date_min.date(),
)
end_picker = widgets.DatePicker(
    description="To:",
    value=None if pd.isna(date_max) else date_max.date(),
)
days_slider = widgets.IntSlider(
    value=2, min=0, max=7, step=1, description="Max days:"
)
amount_slider = widgets.FloatSlider(
    value=1.0, min=0.0, max=20.0, step=0.5, description="Max NIS:"
)
show_unmatched_only = widgets.Checkbox(value=False, description="Only unmatched")
run_button = widgets.Button(description="Run Matching", button_style="primary")
copy_results_button = widgets.Button(description="Copy Results")
export_filtered_button = widgets.Button(description="Export Filtered CSV")
export_matched_button = widgets.Button(description="Export Matched CSV")
controls_out = widgets.Output()
results_out = widgets.Output()

display(
    widgets.VBox([
        widgets.HBox([source_dropdown, show_unmatched_only]),
        widgets.HBox([start_picker, end_picker]),
        widgets.HBox([days_slider, amount_slider]),
        widgets.HBox([run_button, copy_results_button, export_filtered_button, export_matched_button]),
        controls_out,
        results_out,
    ])
)

## 5) Filter Records by Selected Date Window
Filter to a single selected account and inclusive date range.

In [16]:
def filter_single_source_by_date(df: pd.DataFrame, account: str, start_date, end_date) -> pd.DataFrame:
    if start_date is None or end_date is None:
        raise ValueError("Please select both start and end dates.")
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    if end_ts < start_ts:
        raise ValueError("End date must be on or after start date.")

    mask = (df["Account"] == account) & (df["Date_dt"].between(start_ts, end_ts, inclusive="both"))
    filtered = df.loc[mask].copy().reset_index(drop=True)
    return filtered

# Quick validation sample
sample_filtered = filter_single_source_by_date(
    CLEAN_DF, source_dropdown.value, start_picker.value, end_picker.value
)
print(f"Filtered rows for {source_dropdown.value}: {len(sample_filtered)}")
display(sample_filtered.head(10))

Filtered rows for BIT (Rent Securities): 1


,Date,Payee,Memo,Inflow,Outflow,Account,Ownership,source_type,Date_dt,amount_abs,direction
0,10/02/2026,Starting Balance,,0.0,2500.0,BIT (Rent Securities),Shai (Private),ynab,2026-02-10,2500.0,outflow


## 6) Compute Matching Logic Within the Source
Run deterministic matching using configurable tolerances and compute summary metrics.

In [17]:
def run_single_source_matching(full_df: pd.DataFrame, account: str, start_date, end_date, max_days: int, max_amount: float):
    single_source = filter_single_source_by_date(full_df, account, start_date, end_date)
    ynab_side = full_df[
        (full_df["source_type"] == "ynab")
        & (full_df["Account"] == account)
        & (full_df["Date_dt"].between(pd.Timestamp(start_date), pd.Timestamp(end_date), inclusive="both"))
    ].copy()

    if single_source.empty and ynab_side.empty:
        return pd.DataFrame(), {"bank_card_total": 0, "bank_card_matched": 0, "score_pct": 0.0, "ynab_total": 0, "ynab_matched": 0, "by_level": {}}

    match_input = pd.concat([single_source, ynab_side], ignore_index=True)
    levels = [
        MatchLevel(name="exact", date_tolerance=0, amount_tolerance=0.0),
        MatchLevel(name="1_day", date_tolerance=1, amount_tolerance=0.0),
        MatchLevel(name=f"custom_{max_days}d_{max_amount:.1f}nis", date_tolerance=max_days, amount_tolerance=max_amount),
    ]
    matched = run_cascade_matching(match_input, levels=levels)
    summary = compute_match_score(matched)
    return matched, summary

## 7) Map Matched Records Interactively
If latitude/longitude columns exist, plot matched records on a map with color by match status.

In [18]:
def render_match_map(df: pd.DataFrame):
    if not HAS_FOLIUM:
        print("Folium is not installed. Install with: pip install folium")
        return None
    if LAT_COL is None or LON_COL is None:
        print("No latitude/longitude columns found. Skipping map.")
        return None

    map_df = df.dropna(subset=[LAT_COL, LON_COL]).copy()
    if map_df.empty:
        print("No valid coordinates in this filtered result.")
        return None

    center_lat = float(map_df[LAT_COL].mean())
    center_lon = float(map_df[LON_COL].mean())
    m = folium.Map(location=[center_lat, center_lon], zoom_start=9, tiles="OpenStreetMap")
    cluster = MarkerCluster().add_to(m)

    for _, row in map_df.iterrows():
        color = "green" if row.get("match_status") == "matched" else "red"
        popup = (
            f"Date: {row.get('Date','')}<br>"
            f"Account: {row.get('Account','')}<br>"
            f"Inflow: {row.get('Inflow',0)}<br>"
            f"Outflow: {row.get('Outflow',0)}<br>"
            f"Status: {row.get('match_status','')}<br>"
            f"Level: {row.get('match_level','')}<br>"
            f"Payee: {row.get('Payee','')}"
        )
        folium.CircleMarker(
            location=[row[LAT_COL], row[LON_COL]],
            radius=5,
            color=color,
            fill=True,
            fill_opacity=0.8,
            popup=folium.Popup(popup, max_width=380),
        ).add_to(cluster)

    return m

## 8) Display Match Details and Export Results
Run matching from widgets, inspect details, and export filtered/matched outputs.

In [ ]:
OUT_DIR = pathlib.Path("private_data/outgoing")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def _fmt_summary(summary: dict) -> str:
    by_level = summary.get("by_level", {})
    level_text = ", ".join([f"{k}:{v}" for k, v in sorted(by_level.items())]) if by_level else "none"
    return (
        f"bank/card matched: {summary['bank_card_matched']}/{summary['bank_card_total']} | "
        f"YNAB matched: {summary['ynab_matched']}/{summary['ynab_total']} | "
        f"score: {summary['score_pct']}% | levels: {level_text}"
    )

def on_run_clicked(_):
    global LAST_FILTERED, LAST_MATCHED
    with results_out:
        clear_output(wait=True)
        try:
            matched, summary = run_single_source_matching(
                CLEAN_DF,
                source_dropdown.value,
                start_picker.value,
                end_picker.value,
                int(days_slider.value),
                float(amount_slider.value),
            )

            if matched.empty:
                print("No rows found for the selected source/date range.")
                LAST_FILTERED = pd.DataFrame()
                LAST_MATCHED = pd.DataFrame()
                return

            LAST_FILTERED = matched.copy()
            if show_unmatched_only.value:
                view_df = matched[matched["match_status"] == "unmatched"].copy()
            else:
                view_df = matched.copy()
            LAST_MATCHED = view_df.copy()

            print(_fmt_summary(summary))
            cols = [
                "Date", "Account", "Inflow", "Outflow", "source_type",
                "Payee", "Memo", "match_status", "match_level"
            ]
            existing_cols = [c for c in cols if c in view_df.columns]
            display(view_df[existing_cols].sort_values(["Date", "Account"]).reset_index(drop=True).head(500))

            m = render_match_map(view_df)
            if m is not None:
                display(m)
        except Exception as exc:
            print(f"Error: {exc}")

def _copy_df_to_clipboard(df: pd.DataFrame):
    import json
    from IPython.display import Javascript

    if df is None or df.empty:
        print("Nothing to copy. Run matching first.")
        return

    cols = [
        "Date", "Account", "Inflow", "Outflow", "source_type",
        "Payee", "Memo", "match_status", "match_level"
    ]
    existing_cols = [c for c in cols if c in df.columns]
    copy_df = df[existing_cols].sort_values(["Date", "Account"]).reset_index(drop=True)
    tsv_text = copy_df.to_csv(sep="\t", index=False)

    display(Javascript(
        "navigator.clipboard.writeText(" + json.dumps(tsv_text) + ")"
        ".then(() => console.log('Clipboard copy succeeded'))"
        ".catch((err) => console.error('Clipboard copy failed', err));"
    ))
    print(f"Copied {len(copy_df)} rows to clipboard (TSV).")

def _export_df(df: pd.DataFrame, prefix: str):
    if df is None or df.empty:
        print(f"Nothing to export for {prefix}.")
        return
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = OUT_DIR / f"{prefix}_{ts}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Exported {len(df)} rows -> {path}")

def on_copy_results_clicked(_):
    with controls_out:
        clear_output(wait=True)
        _copy_df_to_clipboard(LAST_MATCHED)

def on_export_filtered_clicked(_):
    with controls_out:
        clear_output(wait=True)
        _export_df(LAST_FILTERED, "single_source_filtered")

def on_export_matched_clicked(_):
    with controls_out:
        clear_output(wait=True)
        _export_df(LAST_MATCHED, "single_source_matched_view")

run_button.on_click(on_run_clicked)
copy_results_button.on_click(on_copy_results_clicked)
export_filtered_button.on_click(on_export_filtered_clicked)
export_matched_button.on_click(on_export_matched_clicked)

print("Controls are ready. Select options above and click 'Run Matching'.")

Controls are ready. Select options above and click 'Run Matching'.
